In [1]:
# ============================================================
# PS S6E5 - exp_20260503_006c_cat_original_prior_no_count_min
# CatBoost + original prior minimal features
# No Normalized_TyreLife / No Year prior / No count features
# ============================================================

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier, Pool

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260503_006c_cat_original_prior_no_count_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_BASE_CANDIDATES = [
        Path("/kaggle/input/f1-strategy-dataset-pit-stop-prediction"),
        Path("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction"),
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    SEED = 42
    N_SPLITS = 5

    PARAMS = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 5000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "random_seed": SEED,
        "early_stopping_rounds": 300,
        "verbose": 300,
        "allow_writing_files": False,
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Load competition data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sub shape  :", sub.shape)

print("\ntrain columns:")
print(train.columns.tolist())

print("\ntest columns:")
print(test.columns.tolist())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET not in test.columns

feature_cols_raw = [c for c in train.columns if c not in [CFG.ID_COL, CFG.TARGET]]

print("\nraw feature cols:", len(feature_cols_raw))
print(feature_cols_raw)

print("\ntarget distribution:")
print(train[CFG.TARGET].value_counts(dropna=False))
print(train[CFG.TARGET].value_counts(normalize=True, dropna=False))

train shape: (439140, 16)
test shape : (188165, 15)
sub shape  : (188165, 2)

train columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']

test columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

raw feature cols: 14
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

target distribution:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [5]:
# ============================================================
# Locate and load original dataset
# ============================================================

all_csvs = sorted(Path("/kaggle/input").glob("**/*.csv"))

orig_base = None
for p in CFG.ORIG_BASE_CANDIDATES:
    if p.exists():
        orig_base = p
        break

if orig_base is not None:
    orig_csvs = sorted(orig_base.glob("*.csv"))
else:
    orig_csvs = [
        p for p in all_csvs
        if "playground-series-s6e5" not in str(p)
    ]

print("\noriginal csv candidates:")
for p in orig_csvs:
    print(p)

orig_infos = []
for p in orig_csvs:
    try:
        tmp = pd.read_csv(p, nrows=5)
        cols = tmp.columns.tolist()
        overlap = len((set(train.columns) - {CFG.ID_COL}) & set(cols))
        orig_infos.append({
            "path": str(p),
            "filename": p.name,
            "ncols": len(cols),
            "overlap": overlap,
            "has_target": CFG.TARGET in cols,
            "has_danger": CFG.DANGER_COL in cols,
            "columns": cols,
        })
    except Exception as e:
        print("failed:", p, e)

assert len(orig_infos) > 0, "original csv candidate was not found."

orig_infos = sorted(
    orig_infos,
    key=lambda x: (x["has_target"], x["overlap"], x["ncols"]),
    reverse=True,
)

orig_path = Path(orig_infos[0]["path"])
print("\nselected original:", orig_path)

orig = pd.read_csv(orig_path)

print("original shape:", orig.shape)
print("original columns:")
print(orig.columns.tolist())

assert CFG.TARGET in orig.columns, "originalにtargetがありません"
assert CFG.DANGER_COL in orig.columns, "想定ではNormalized_TyreLifeがoriginalに存在するはずです"

# 危険列は即削除。以降の処理で絶対に参照しない。
orig = orig.drop(columns=[CFG.DANGER_COL])

print("\noriginal columns after dropping danger col:")
print(orig.columns.tolist())

train_cols_no_id = set(train.columns) - {CFG.ID_COL}
orig_cols = set(orig.columns)

print("\ntrain only vs original after dropping danger:")
print(sorted(train_cols_no_id - orig_cols))

print("\noriginal only vs train after dropping danger:")
print(sorted(orig_cols - train_cols_no_id))

assert sorted(train_cols_no_id - orig_cols) == [], "train側にoriginalへ対応しない列があります"
assert sorted(orig_cols - train_cols_no_id) == [], "danger除外後もoriginal側に余分な列があります"


original csv candidates:
/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv

selected original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
original shape: (101371, 16)
original columns:
['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'Race', 'Year', 'LapTime_Delta', 'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress', 'Normalized_TyreLife', 'Position_Change']

original columns after dropping danger col:
['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'Race', 'Year', 'LapTime_Delta', 'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress', 'Position_Change']

train only vs original after dropping danger:
[]

original only vs train after dropping danger:
[]


In [6]:
# ============================================================
# Feature Engineering: original prior no Year prior / no count
# ============================================================

def add_original_prior_features_no_count(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    target_col: str,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    """
    Add minimal original target prior features.

    006c policy:
    - Use original target only.
    - Do not use competition train target.
    - Do not use Normalized_TyreLife.
    - Do not merge original rows into training.
    - Do not create Year original prior.
    - Do not create *_count features.
    """

    train_df = train_df.copy()
    test_df = test_df.copy()
    orig_df = orig_df.copy()

    global_prior = float(orig_df[target_col].mean())
    print("\noriginal global prior:", global_prior)
    print("competition train target mean:", float(train_df[target_col].mean()))

    added = []
    prior_info = {
        "global_prior": global_prior,
        "categorical_prior_cols": [],
        "numeric_bin_prior_cols": [],
        "pair_prior_cols": [],
        "removed_from_006": [
            "ORIG_PRIOR_Year",
            "ORIG_PRIOR_Year_count",
            "ORIG_PRIOR_Year_minus_global",
        ],
        "removed_from_006b": [
            "*_count",
        ],
        "count_features_used": False,
        "year_prior_used": False,
    }

    # ----------------------------
    # 1. Categorical original priors
    # ----------------------------
    # 006b/006c: Year prior は除外
    cat_prior_cols = [
        "Driver",
        "Compound",
        "Race",
        # "Year",  # removed from 006b onward
        "Stint",
        "PitStop",
        "Position",
    ]

    for col in cat_prior_cols:
        if col not in orig_df.columns:
            continue
        if col not in train_df.columns or col not in test_df.columns:
            continue

        prior_name = f"ORIG_PRIOR_{col}"

        prior = (
            orig_df
            .groupby(col, dropna=False)[target_col]
            .agg(["mean"])
            .reset_index()
        )
        prior = prior.rename(columns={"mean": prior_name})

        train_df = train_df.merge(prior, on=col, how="left")
        test_df = test_df.merge(prior, on=col, how="left")

        train_df[prior_name] = train_df[prior_name].fillna(global_prior)
        test_df[prior_name] = test_df[prior_name].fillna(global_prior)

        added.append(prior_name)
        prior_info["categorical_prior_cols"].append(col)

    # ----------------------------
    # 2. Numeric bin original priors
    # ----------------------------
    numeric_bin_cols = [
        "LapNumber",
        "TyreLife",
        "RaceProgress",
        "Cumulative_Degradation",
        "LapTime_Delta",
    ]

    # 存在しない列があっても壊れないようにする
    numeric_bin_cols = [
        c for c in numeric_bin_cols
        if c in train_df.columns and c in test_df.columns and c in orig_df.columns
    ]

    combined_for_edges = pd.concat(
        [train_df[numeric_bin_cols], test_df[numeric_bin_cols], orig_df[numeric_bin_cols]],
        axis=0,
        ignore_index=True,
    )

    bin_edges_info = {}

    for col in numeric_bin_cols:
        values = combined_for_edges[col].dropna()
        if values.nunique() <= 2:
            continue

        try:
            quantiles = np.linspace(0, 1, 11)
            edges = np.unique(values.quantile(quantiles).values)

            if len(edges) <= 2:
                continue

            edges[0] = -np.inf
            edges[-1] = np.inf

            bin_col = f"{col}_bin10"
            prior_name = f"ORIG_PRIOR_{bin_col}"

            train_df[bin_col] = pd.cut(train_df[col], bins=edges, include_lowest=True)
            test_df[bin_col] = pd.cut(test_df[col], bins=edges, include_lowest=True)
            orig_df[bin_col] = pd.cut(orig_df[col], bins=edges, include_lowest=True)

            prior = (
                orig_df
                .groupby(bin_col, observed=True)[target_col]
                .agg(["mean"])
                .reset_index()
            )
            prior = prior.rename(columns={"mean": prior_name})

            train_df = train_df.merge(prior, on=bin_col, how="left")
            test_df = test_df.merge(prior, on=bin_col, how="left")

            train_df[prior_name] = train_df[prior_name].fillna(global_prior)
            test_df[prior_name] = test_df[prior_name].fillna(global_prior)

            # CatBoostにInterval dtypeを渡さないため、bin列は削除する
            train_df = train_df.drop(columns=[bin_col])
            test_df = test_df.drop(columns=[bin_col])
            orig_df = orig_df.drop(columns=[bin_col])

            added.append(prior_name)
            prior_info["numeric_bin_prior_cols"].append(col)
            bin_edges_info[col] = [float(x) if np.isfinite(x) else str(x) for x in edges]

        except Exception as e:
            print(f"failed numeric bin prior for {col}:", e)

    prior_info["bin_edges"] = bin_edges_info

    # ----------------------------
    # 3. Very coarse pair priors
    # ----------------------------
    # 細かすぎる Race x Driver x Stint は入れない。
    pair_cols = [
        ("Compound", "Stint"),
        ("Compound", "PitStop"),
        ("Race", "Stint"),
        ("Race", "PitStop"),
    ]

    for c1, c2 in pair_cols:
        if c1 not in orig_df.columns or c2 not in orig_df.columns:
            continue
        if c1 not in train_df.columns or c2 not in train_df.columns:
            continue

        pair_key = f"{c1}__{c2}"
        prior_name = f"ORIG_PRIOR_{pair_key}"

        for df in [train_df, test_df, orig_df]:
            df[pair_key] = df[c1].astype(str) + "__" + df[c2].astype(str)

        prior = (
            orig_df
            .groupby(pair_key, dropna=False)[target_col]
            .agg(["mean"])
            .reset_index()
        )
        prior = prior.rename(columns={"mean": prior_name})

        train_df = train_df.merge(prior, on=pair_key, how="left")
        test_df = test_df.merge(prior, on=pair_key, how="left")

        train_df[prior_name] = train_df[prior_name].fillna(global_prior)
        test_df[prior_name] = test_df[prior_name].fillna(global_prior)

        # pair_key自体はカテゴリ特徴として残さず削除する
        train_df = train_df.drop(columns=[pair_key])
        test_df = test_df.drop(columns=[pair_key])
        orig_df = orig_df.drop(columns=[pair_key])

        added.append(prior_name)
        prior_info["pair_prior_cols"].append([c1, c2])

    # ----------------------------
    # 4. Simple prior deviations
    # ----------------------------
    prior_mean_cols = [
        c for c in added
        if c.startswith("ORIG_PRIOR_") and not c.endswith("_count")
    ]

    for c in prior_mean_cols:
        new_col = f"{c}_minus_global"
        train_df[new_col] = train_df[c] - global_prior
        test_df[new_col] = test_df[c] - global_prior
        added.append(new_col)

    prior_info["added_features"] = added
    prior_info["added_feature_count"] = len(added)

    return train_df, test_df, added, prior_info


train_fe, test_fe, added_features, prior_info = add_original_prior_features_no_count(
    train_df=train,
    test_df=test,
    orig_df=orig,
    target_col=CFG.TARGET,
)

print("\nadded features:", len(added_features))
for c in added_features:
    print(c)

# 安全確認: Year prior と count が生成されていないこと
for c in [
    "ORIG_PRIOR_Year",
    "ORIG_PRIOR_Year_count",
    "ORIG_PRIOR_Year_minus_global",
]:
    assert c not in added_features, f"{c} should not be generated in 006c"
    assert c not in train_fe.columns, f"{c} should not exist in train_fe"
    assert c not in test_fe.columns, f"{c} should not exist in test_fe"

count_cols = [c for c in added_features if c.endswith("_count")]
assert len(count_cols) == 0, f"count features should not be generated in 006c: {count_cols}"

train_count_cols = [c for c in train_fe.columns if c.startswith("ORIG_PRIOR_") and c.endswith("_count")]
test_count_cols = [c for c in test_fe.columns if c.startswith("ORIG_PRIOR_") and c.endswith("_count")]
assert len(train_count_cols) == 0, f"count features exist in train_fe: {train_count_cols}"
assert len(test_count_cols) == 0, f"count features exist in test_fe: {test_count_cols}"

print("\n006c safety checks passed.")
print("No ORIG_PRIOR_Year features.")
print("No *_count features.")

print("\nprior info:")
print(json.dumps(prior_info, ensure_ascii=False, indent=2)[:5000])

feature_cols = [c for c in train_fe.columns if c not in [CFG.ID_COL, CFG.TARGET]]
print("\ntotal feature cols:", len(feature_cols))


original global prior: 0.25479673673930414
competition train target mean: 0.19898210137996994

added features: 30
ORIG_PRIOR_Driver
ORIG_PRIOR_Compound
ORIG_PRIOR_Race
ORIG_PRIOR_Stint
ORIG_PRIOR_PitStop
ORIG_PRIOR_Position
ORIG_PRIOR_LapNumber_bin10
ORIG_PRIOR_TyreLife_bin10
ORIG_PRIOR_RaceProgress_bin10
ORIG_PRIOR_Cumulative_Degradation_bin10
ORIG_PRIOR_LapTime_Delta_bin10
ORIG_PRIOR_Compound__Stint
ORIG_PRIOR_Compound__PitStop
ORIG_PRIOR_Race__Stint
ORIG_PRIOR_Race__PitStop
ORIG_PRIOR_Driver_minus_global
ORIG_PRIOR_Compound_minus_global
ORIG_PRIOR_Race_minus_global
ORIG_PRIOR_Stint_minus_global
ORIG_PRIOR_PitStop_minus_global
ORIG_PRIOR_Position_minus_global
ORIG_PRIOR_LapNumber_bin10_minus_global
ORIG_PRIOR_TyreLife_bin10_minus_global
ORIG_PRIOR_RaceProgress_bin10_minus_global
ORIG_PRIOR_Cumulative_Degradation_bin10_minus_global
ORIG_PRIOR_LapTime_Delta_bin10_minus_global
ORIG_PRIOR_Compound__Stint_minus_global
ORIG_PRIOR_Compound__PitStop_minus_global
ORIG_PRIOR_Race__Stint_minus

In [7]:
# ============================================================
# Column types
# ============================================================

# 002 CatBoost RAW baseline に合わせる。
cat_cols = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "LapNumber",
    "PitStop",
    "Position",
    "Stint",
]
cat_cols = [c for c in cat_cols if c in feature_cols]

num_cols = [c for c in feature_cols if c not in cat_cols]

print("\ncat_cols:", len(cat_cols), cat_cols)
print("num_cols:", len(num_cols), num_cols)


cat_cols: 8 ['Driver', 'Compound', 'Race', 'Year', 'LapNumber', 'PitStop', 'Position', 'Stint']
num_cols: 36 ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'ORIG_PRIOR_Driver', 'ORIG_PRIOR_Compound', 'ORIG_PRIOR_Race', 'ORIG_PRIOR_Stint', 'ORIG_PRIOR_PitStop', 'ORIG_PRIOR_Position', 'ORIG_PRIOR_LapNumber_bin10', 'ORIG_PRIOR_TyreLife_bin10', 'ORIG_PRIOR_RaceProgress_bin10', 'ORIG_PRIOR_Cumulative_Degradation_bin10', 'ORIG_PRIOR_LapTime_Delta_bin10', 'ORIG_PRIOR_Compound__Stint', 'ORIG_PRIOR_Compound__PitStop', 'ORIG_PRIOR_Race__Stint', 'ORIG_PRIOR_Race__PitStop', 'ORIG_PRIOR_Driver_minus_global', 'ORIG_PRIOR_Compound_minus_global', 'ORIG_PRIOR_Race_minus_global', 'ORIG_PRIOR_Stint_minus_global', 'ORIG_PRIOR_PitStop_minus_global', 'ORIG_PRIOR_Position_minus_global', 'ORIG_PRIOR_LapNumber_bin10_minus_global', 'ORIG_PRIOR_TyreLife_bin10_minus_global', 'ORIG_PRIOR_RaceProgress_bin10_minus_global', 'ORIG_PRIOR_Cumulative_Degradation

In [8]:
# ============================================================
# Prepare X/y
# ============================================================

X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()

# CatBoost categorical columns must be string-like with missing filled.
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("__MISSING__")
    X_test[c] = X_test[c].astype("string").fillna("__MISSING__")

assert train_fe[CFG.TARGET].notna().all()
assert set(train_fe[CFG.TARGET].dropna().unique()).issubset({0, 1, 0.0, 1.0})
y = train_fe[CFG.TARGET].astype(int).values

cat_features = [X.columns.get_loc(c) for c in cat_cols]

print("\nX shape:", X.shape)
print("X_test shape:", X_test.shape)
print("cat_features indices:", cat_features)

print("\nNaN count in X numeric part:", int(X[num_cols].isna().sum().sum()))
print("NaN count in X_test numeric part:", int(X_test[num_cols].isna().sum().sum()))


X shape: (439140, 44)
X_test shape: (188165, 44)
cat_features indices: [0, 1, 2, 3, 5, 4, 8, 6]

NaN count in X numeric part: 0
NaN count in X_test numeric part: 0


In [9]:
# ============================================================
# CV Training
# ============================================================

oof = np.zeros(len(train_fe), dtype=float)
pred = np.zeros(len(test_fe), dtype=float)

fold_scores = []
fold_best_iterations = []
feature_importance_list = []

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}")
    print("=" * 80)

    X_tr = X.iloc[tr_idx]
    X_va = X.iloc[va_idx]
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_va, y_va, cat_features=cat_features)
    test_pool = Pool(X_test, cat_features=cat_features)

    model = CatBoostClassifier(**CFG.PARAMS)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    va_pred = model.predict_proba(valid_pool)[:, 1]
    te_pred = model.predict_proba(test_pool)[:, 1]

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(float(auc))
    fold_best_iterations.append(int(model.get_best_iteration()))

    print(f"fold {fold} AUC: {auc:.8f}")
    print(f"fold {fold} best_iteration: {model.get_best_iteration()}")

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.get_feature_importance(),
        "fold": fold,
    })
    feature_importance_list.append(fi)


cv_auc = roc_auc_score(y, oof)

print("\n" + "=" * 80)
print("CV RESULT")
print("=" * 80)
print("CV AUC:", cv_auc)
print("fold_scores:", fold_scores)
print("mean:", np.mean(fold_scores))
print("std :", np.std(fold_scores))
print("fold_best_iterations:", fold_best_iterations)


Fold 1
0:	test: 0.9081212	best: 0.9081212 (0)	total: 547ms	remaining: 45m 35s
300:	test: 0.9379747	best: 0.9379747 (300)	total: 1m 50s	remaining: 28m 42s
600:	test: 0.9427889	best: 0.9427889 (600)	total: 3m 40s	remaining: 26m 57s
900:	test: 0.9449342	best: 0.9449342 (900)	total: 5m 33s	remaining: 25m 16s
1200:	test: 0.9462155	best: 0.9462155 (1200)	total: 7m 25s	remaining: 23m 28s
1500:	test: 0.9469867	best: 0.9469867 (1500)	total: 9m 16s	remaining: 21m 38s
1800:	test: 0.9475526	best: 0.9475526 (1800)	total: 11m 6s	remaining: 19m 43s
2100:	test: 0.9479884	best: 0.9479884 (2100)	total: 12m 56s	remaining: 17m 51s
2400:	test: 0.9483207	best: 0.9483217 (2399)	total: 14m 47s	remaining: 16m
2700:	test: 0.9486031	best: 0.9486031 (2700)	total: 16m 37s	remaining: 14m 9s
3000:	test: 0.9488100	best: 0.9488100 (3000)	total: 18m 29s	remaining: 12m 18s
3300:	test: 0.9489803	best: 0.9489803 (3300)	total: 20m 21s	remaining: 10m 28s
3600:	test: 0.9491617	best: 0.9491617 (3600)	total: 22m 12s	remaining

In [10]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

sub_out = sub.copy()
pred_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[pred_col] = pred
sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

feature_df = pd.DataFrame({
    "feature": feature_cols,
    "is_added": [c in added_features for c in feature_cols],
    "is_categorical": [c in cat_cols for c in feature_cols],
    "is_count_feature": [c.endswith("_count") for c in feature_cols],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

fi_all = pd.concat(feature_importance_list, axis=0, ignore_index=True)
fi_summary = (
    fi_all
    .groupby("feature", as_index=False)["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

with open(CFG.OUTDIR / "prior_info.json", "w", encoding="utf-8") as f:
    json.dump(prior_info, f, ensure_ascii=False, indent=2)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "original_shape_after_danger_drop": list(orig.shape),
        "selected_original_path": str(orig_path),
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "competition_target_mean": float(train[CFG.TARGET].mean()),
        "original_target_mean_after_danger_drop": float(orig[CFG.TARGET].mean()),
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
    },
    "features": {
        "raw_feature_count": len(feature_cols_raw),
        "total_feature_count": len(feature_cols),
        "added_feature_count": len(added_features),
        "categorical_features": cat_cols,
        "numerical_features": num_cols,
        "added_features": added_features,
        "prior_info": prior_info,
        "notes": [
            "Original dataset is used only for target prior features.",
            "Normalized_TyreLife is dropped before any feature engineering.",
            "No Normalized_TyreLife proxy is created.",
            "Original rows are not merged into training.",
            "Competition train target is not used for target encoding.",
            "006c removes Year original prior features inherited from 006b.",
            "006c removes all *_count features from original prior blocks.",
            "Coarse categorical, numeric-bin, and pair priors are kept as mean and minus_global only.",
        ],
    },
    "model": {
        "family": "CatBoost",
        "params": CFG.PARAMS,
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "fold_scores": fold_scores,
        "fold_best_iterations": fold_best_iterations,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
    },
    "safety_checks": {
        "has_ORIG_PRIOR_Year": "ORIG_PRIOR_Year" in feature_cols,
        "has_ORIG_PRIOR_Year_count": "ORIG_PRIOR_Year_count" in feature_cols,
        "has_ORIG_PRIOR_Year_minus_global": "ORIG_PRIOR_Year_minus_global" in feature_cols,
        "count_feature_count": int(sum(c.endswith("_count") for c in feature_cols)),
        "count_features": [c for c in feature_cols if c.endswith("_count")],
    },
    "artifacts": {
        "oof": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "submission": str(sub_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
        "prior_info": str(CFG.OUTDIR / "prior_info.json"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved artifacts to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(fi_summary.head(80))
display(sub_out.head())


Saved artifacts to: /kaggle/working/exp_20260503_006c_cat_original_prior_no_count_min
Submission: /kaggle/working/exp_20260503_006c_cat_original_prior_no_count_min/submission_exp_20260503_006c_cat_original_prior_no_count_min.csv


,index,feature,mean,std
43,43,Year,34.803893,1.317450
42,42,TyreLife,7.834372,0.288897
40,40,RaceProgress,5.194002,0.303999
5,5,LapTime_Delta,3.537464,0.179570
10,10,ORIG_PRIOR_Compound__Stint_minus_global,3.381204,0.634464
41,41,Stint,3.226591,0.343597
1,1,Cumulative_Degradation,3.143610,0.039879
38,38,Position_Change,3.017540,0.202472
9,9,ORIG_PRIOR_Compound__Stint,2.986934,0.474507
4,4,LapTime (s),2.849328,0.167706


,id,PitNextLap
0,439140,0.005131
1,439141,0.003309
2,439142,0.004987
3,439143,0.115700
4,439144,0.811071
